# TinyChirp LEAF-Time TensorFlow

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from typing import TYPE_CHECKING

from utils import (
    TARGET_AUDIO_LEN_TIME,
    get_paths,
    configure_tf_runtime,
    set_global_seed,
    make_time_datasets,
    build_representative_batches,
    export_keras_model_to_int8_tflite,
)

if TYPE_CHECKING:
    import keras
else:
    from tensorflow import keras

configure_tf_runtime()
set_global_seed()

MODEL_STEM = "leaf_multilayer_tf"
paths = get_paths(MODEL_STEM)
OUT_TFLITE = paths.out_tflite
OUT_AUDIO_RS = paths.out_audio_rs


Dataset root: /home/nathan/Documents/tiny-chirp-microflow/dataset
Model output: /home/nathan/Documents/tiny-chirp-microflow/models/leaf_tf.tflite
Audio sample output: /home/nathan/Documents/tiny-chirp-microflow/src/audio_sample.rs
Sample rate: 16000
Frame length: 1024
Frame step: 256
Target frames (time): 184
Target audio length (time): 47872


In [ ]:
train_ds, val_ds, test_ds, label_names = make_time_datasets()
num_labels = len(label_names)
print("Classes:", label_names)


Found 11292 files belonging to 2 classes.
Found 1380 files belonging to 2 classes.
Found 1393 files belonging to 2 classes.
Classes: ['non_target' 'target']
Num labels: 2


In [ ]:
import numpy as np
import tensorflow as tf
import keras
from model_parts import GaborConv1D, GaussianPool1D, LogCompression

In [ ]:
NUM_FILTERS = 48
KERNEL_SIZE = 64
STRIDE = 16

NUM_FILTERS_2 = 16
KERNEL_SIZE_2 = 16
STRIDE_2 = 4

def build_training_model(num_labels: int) -> keras.Model:
    inputs = keras.Input(shape=(TARGET_AUDIO_LEN_TIME, 1))

    x = GaborConv1D(num_filters=NUM_FILTERS, kernel_size=KERNEL_SIZE, stride=STRIDE, name="gabor_conv")(inputs)
    x = GaussianPool1D(num_filters=NUM_FILTERS, pool_size=KERNEL_SIZE, stride=1, name="gauss_pool")(x)
    x = keras.layers.Conv1D(
        filters=NUM_FILTERS_2, 
        kernel_size=KERNEL_SIZE_2, 
        strides=STRIDE_2,
        padding="same", 
        use_bias=False, 
        name="conv_1d_2")(x)
    x = keras.layers.ReLU(name="relu")(x)


    x = LogCompression(name="log_compress")(x)
    
    x = keras.layers.GlobalAveragePooling1D()(x)
    x = keras.layers.Dense(64, activation='relu')(x)
    outputs = keras.layers.Dense(num_labels, activation=None, name="dense_logits")(x)
    
    return keras.Model(inputs, outputs, name="leaf_fast_bird_model")


num_labels = 2 
training_model = build_training_model(num_labels)

# Use XLA compilation to speed up training
training_model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
    jit_compile=True  # This forces TF to optimize the custom math loops
)

In [ ]:
training_model.summary()

Model: "leaf_fast_bird_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 47872, 1)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gabor_conv (GaborConv1D)        │ (None, 2992, 48)       │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gauss_pool (GaussianPool1D)     │ (None, 2992, 48)       │            48 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_1d_2 (Conv1D)              │ (None, 748, 16)        │        12,288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu (ReLU)                     │ (None, 748, 16)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ log_compress (LogCompression)   │ (None, 748, 16)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 16)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_logits (Dense)            │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,650 (53.32 KB)

 Trainable params: 13,650 (53.32 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from utils import init_wandb, get_callbacks, finish_wandb

init_wandb(MODEL_STEM, config={
    "num_filters": NUM_FILTERS,
    "kernel_size": KERNEL_SIZE,
    "stride": STRIDE,
})

history = training_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    validation_steps=50,
    callbacks=get_callbacks(10,5,32),
)
finish_wandb()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/nathan/.netrc.
wandb: Currently logged in as: nathan-duboisset (nathan-duboisset-cole-polytechnique) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1/100


I0000 00:00:1776360322.220839  117231 service.cc:145] XLA service 0x70e31c049d60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776360322.220869  117231 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9


  7/353 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.4598 - loss: 3.7859

I0000 00:00:1776360325.011884  117231 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


353/353 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.6716 - loss: 0.7513

/home/nathan/Documents/tiny-chirp-microflow/building_tensorflow/.venv/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


353/353 ━━━━━━━━━━━━━━━━━━━━ 25s 60ms/step - accuracy: 0.7222 - loss: 0.5328 - val_accuracy: 0.8464 - val_loss: 0.4081
Epoch 2/100
353/353 ━━━━━━━━━━━━━━━━━━━━ 19s 53ms/step - accuracy: 0.8533 - loss: 0.3479 - val_accuracy: 0.9232 - val_loss: 0.2376
Epoch 3/100
353/353 ━━━━━━━━━━━━━━━━━━━━ 19s 54ms/step - accuracy: 0.8972 - loss: 0.2526 - val_accuracy: 0.9196 - val_loss: 0.1814
Epoch 4/100
353/353 ━━━━━━━━━━━━━━━━━━━━ 19s 54ms/step - accuracy: 0.9160 - loss: 0.2139 - val_accuracy: 0.8949 - val_loss: 0.2172
Epoch 5/100
353/353 ━━━━━━━━━━━━━━━━━━━━ 20s 56ms/step - accuracy: 0.9236 - loss: 0.1929 - val_accuracy: 0.9297 - val_loss: 0.1521
Epoch 6/100
353/353 ━━━━━━━━━━━━━━━━━━━━ 20s 55ms/step - accuracy: 0.9255 - loss: 0.1878 - val_accuracy: 0.9290 - val_loss: 0.1678
Epoch 7/100
353/353 ━━━━━━━━━━━━━━━━━━━━ 19s 55ms/step - accuracy: 0.9332 - loss: 0.1665 - val_accuracy: 0.9413 - val_loss: 0.1744
Epoch 8/100
353/353 ━━━━━━━━━━━━━━━━━━━━ 19s 55ms/step - accuracy: 0.9393 - loss: 0.1512 - val_

batch/accuracy,▁▃▃▃▄▃▄▄▆▆▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▇▇█
batch/batch_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇███
batch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/loss,█▅▄▃▃▃▂▂▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,▁▅▆▆▆▇▇▇▇▇▇▇█████████████████████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇███
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▅▄▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▅▅▃▅▅▆▇▆▇▇▇▇▇▆██▇▇█▇███▇█▇█▅▇███
epoch/val_loss,█▅▄▄▃▃▄▂▃▂▂▂▂▁▃▁▁▂▂▁▁▁▁▁▂▁▁▁▃▁▁▁▁
+6,...


In [ ]:
from model_parts import SquaredModulus

gabor_layer = training_model.get_layer("gabor_conv")
pool_layer = training_model.get_layer("gauss_pool")

baked_gabor = gabor_layer.get_filters().numpy()
baked_gauss = pool_layer.get_filters().numpy()

infer_inputs = keras.Input(shape=(TARGET_AUDIO_LEN_TIME, 1))

x = keras.layers.Conv1D(
    filters=NUM_FILTERS * 2,
    kernel_size=KERNEL_SIZE,
    strides=STRIDE,
    padding="same",
    use_bias=False,
    name="baked_gabor_conv",
)(infer_inputs)
x = SquaredModulus(name="squared_modulus")(x)
x = keras.layers.DepthwiseConv1D(
    kernel_size=KERNEL_SIZE,
    strides=1,
    padding="same",
    use_bias=False,
    name="baked_gauss_dw",
)(x)


x = training_model.get_layer("conv_1d_2")(x)
x = training_model.get_layer("relu")(x)
x = training_model.get_layer("log_compress")(x)
x = training_model.get_layer("global_average_pooling1d")(x)
x = training_model.get_layer("dense")(x)
x = training_model.get_layer("dense_logits")(x)

inference_model = keras.Model(infer_inputs, x, name="leaf_fast_bird_inference")

inference_model.get_layer("baked_gabor_conv").set_weights([baked_gabor])
inference_model.get_layer("baked_gauss_dw").set_weights([baked_gauss])

for batch_audio, _ in test_ds.take(1):
    batch_audio_np = batch_audio.numpy()
    logits_train = training_model.predict(batch_audio_np, verbose=0)
    logits_infer = inference_model.predict(batch_audio_np, verbose=0)
    print(f"Max abs diff: {np.max(np.abs(logits_train - logits_infer)):.8f}")

In [ ]:
from utils import get_flops_native
flops = get_flops_native(inference_model, batch_size=1)
print(f"Total FLOPs: {flops}")

Total FLOPs: 74000514


/home/nathan/Documents/tiny-chirp-microflow/building_tensorflow/.venv/lib/python3.11/site-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: keras_tensor_9
Received: inputs=['Tensor(shape=(1, 47872, 1))']
  warnings.warn(msg)


In [ ]:
rep_batches = build_representative_batches(test_ds, take=100)
export_keras_model_to_int8_tflite(inference_model, rep_batches, OUT_TFLITE)
print(f"Success! Wrote {OUT_TFLITE}")

Saved artifact at 'temp_saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 47872, 1), dtype=tf.float32, name='keras_tensor_9')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  124121741190320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  124121741190496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  124123807529296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  124123807530176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  124123807529648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  124123807530000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  124123807530528: TensorSpec(shape=(), dtype=tf.resource, name=None)


W0000 00:00:1776361017.014864  117135 tf_tfl_flatbuffer_helpers.cc:390] Ignored output_format.
W0000 00:00:1776361017.014878  117135 tf_tfl_flatbuffer_helpers.cc:393] Ignored drop_control_dependency.


Success! Wrote /home/nathan/Documents/tiny-chirp-microflow/models/leaf_tf.tflite


fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8


In [ ]:
from utils import evaluate_tflite_model, SAMPLE_RATE

hyperparams = {
    "num_filters": NUM_FILTERS,
    "kernel_size": KERNEL_SIZE,
    "stride": STRIDE,
    "num_filters_2": NUM_FILTERS_2,
    "kernel_size_2": KERNEL_SIZE_2,
    "stride_2": STRIDE_2,
    "dense_hidden": 64,
    "target_audio_len": TARGET_AUDIO_LEN_TIME,
    "sample_rate": SAMPLE_RATE,
    "batch_size": 32,
}

train_m, test_m, avg_ms = evaluate_tflite_model(
    OUT_TFLITE, MODEL_STEM, train_ds, test_ds, hyperparams=hyperparams,
)
print(f"Avg inference: {avg_ms:.3f} ms")